<a href="https://colab.research.google.com/github/chocbagel/Shift-Management-System-LowCode/blob/master/shift-management.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

scope = ['https://spreadsheets.google.com/feeds', 'https://www.googleapis.com/auth/drive']
creds = ServiceAccountCredentials.from_json_keyfile_name('shift-management-credentials.json', scope)
client = gspread.authorize(creds)

sheet = client.open('ShiftManagementSystem').worksheet('DemandData')

data = sheet.get_all_records()
df_sheet = pd.DataFrame(data)

print("training...")
np.random.seed(42)
historical_dates = pd.date_range(start="2025-08-01", end="2026-01-31", freq='D')
train_data = []
# weekend，12pm and 18pm, Server
for d in historical_dates:
    dow = d.dayofweek
    is_wknd = 1 if dow >= 5 else 0
    for h in [8, 12, 18]:
        for p in ['Server', 'Cook', 'Cashier', 'Apprentie']:
            base = 2 if is_wknd else 1
            if h in [12, 18]: base += 2
            if p == 'Server': base += 1
            elif p == 'Apprentie': base -= 1
            demand = max(1, base + np.random.randint(-1, 2))
            train_data.append({'DayOfWeek': dow, 'IsWeekend': is_wknd, 'Hour': h, 'Position': p, 'ActualDemand': demand})

df_train = pd.DataFrame(train_data)
# One-Hot Encoding
df_train_encoded = pd.get_dummies(df_train, columns=['Position'])

X = df_train_encoded.drop('ActualDemand', axis=1)
y = df_train_encoded['ActualDemand']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

y_pred_test = model.predict(X_test)
print(f"\n--- Model Evaluation Metrics ---")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, y_pred_test)):.2f}")
print(f"MAE : {mean_absolute_error(y_test, y_pred_test):.2f}")
print(f"R-squared (R2): {r2_score(y_test, y_pred_test):.2f}")


print("\n Predicting...")
df_sheet['Date'] = pd.to_datetime(df_sheet['Date'])
df_sheet['DayOfWeek'] = df_sheet['Date'].dt.dayofweek
df_sheet['IsWeekend'] = df_sheet['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

X_predict = df_sheet[['DayOfWeek', 'IsWeekend', 'Hour', 'Position']]
X_predict_encoded = pd.get_dummies(X_predict, columns=['Position'])

X_predict_encoded = X_predict_encoded.reindex(columns=X_train.columns, fill_value=0)

predictions = model.predict(X_predict_encoded)
predictions_int = np.round(predictions).astype(int)

pred_values = [[int(p)] for p in predictions_int]
cell_range = f'E2:E{len(predictions_int)+1}'

sheet.update(values=pred_values, range_name=cell_range)
print("\n successed!")

training...

--- Model Evaluation Metrics ---
RMSE : 0.75
MAE : 0.63
R-squared (R2): 0.69

 Predicting...

 successed!
